# Scanned PDF — Single Pass Extraction

Extract structured data from a **scanned** (image-based) PDF in a single LLM call.

The scanned strategy uses a vision LLM to parse page images — useful for PDFs without an embedded text layer (e.g. scanned documents, photos of pages).

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
import sys

from pydantic import BaseModel

from doc_intelligence import PDFExtractionMode, PDFProcessor
from doc_intelligence.pdf.types import ParseStrategy

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from utils import show_pdf_with_bboxes

PDF_PATH = "/Users/zeel/Public/ms/open_source/document_ai/data/sample_invoice.pdf"
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
class Output(BaseModel):
    blue_shield_charge: float

In [4]:
processor = PDFProcessor(
    provider="openai",
    model="gpt-5.4",
    include_citations=True,
    extraction_mode=PDFExtractionMode.SINGLE_PASS,
    strategy=ParseStrategy.SCANNED,
)

In [5]:
response = processor.extract(PDF_PATH, Output, page_numbers=[0, 1])

2026-04-06 23:49:07.937 | DEBUG    | doc_intelligence.pdf.parser:_parse_scanned_vlm:231 - VLM parse: batch starting at page 0, 1 page(s)
2026-04-06 23:49:07.937 | DEBUG    | doc_intelligence.llm:generate:29 - OpenAILLM: generate: 1 image(s), model=gpt-5.4
2026-04-06 23:49:18.817 | INFO     | doc_intelligence.pdf.processor:extract:71 - Document parsed successfully
2026-04-06 23:49:18.817 | DEBUG    | doc_intelligence.pdf.extractor:_run_single_pass:112 - PDFExtractor: extract: json_instance_schema: {
    "blue_shield_charge": {
        "value": <number>,
        "citations": [{"page": <integer>, "blocks": [<integer>]}]
    }
}
2026-04-06 23:49:18.818 | INFO     | doc_intelligence.pdf.formatter:format_document_for_llm:102 - Formatting 1 pages
2026-04-06 23:49:18.818 | DEBUG    | doc_intelligence.pdf.extractor:_run_single_pass:120 - PDFExtractor: extract: content_text: <page number="0">
<block index="0" type="table">
| DATE | CODE | DESCRIPTION OF SERVICE | CHARGE | ADJUSTMENT | INS PYMT |

In [6]:
response

ExtractionResult(data=Output(blue_shield_charge=300.0), metadata={'blue_shield_charge': {'value': 300.0, 'citations': [{'page': 0, 'bboxes': [{'x0': 0.016524216524216526, 'top': 0.0, 'x1': 0.9612535612535612, 'bottom': 0.9217741935483871}]}]}})

In [7]:
assert response.metadata is not None
show_pdf_with_bboxes(
    PDF_PATH,
    response.metadata["blue_shield_charge"]["citations"][0],
    out_file=os.path.join(OUT_DIR, "scanned_single_pass_bbox.png"),
)

  -> saved to /Users/zeel/Public/ms/open_source/document_ai/notebooks/pdf/output/scanned_single_pass_bbox.png
